# Лабораторная работа 1.5. QR-разложение и QR-алгоритм

**Задание.** Реализовать алгоритм QR-разложения матриц в виде программы. На его основе разработать программу,
реализующую QR-алгоритм решения полной проблемы собственных значений произвольных матриц, задавая в качестве
входных данных матрицу и точность вычислений. С использованием разработанного ПО найти собственные значения матрицы.

$$
A = \begin{pmatrix} -2 & 7 & -6 \\ -1 & 9 & -4 \\ -1 & 8 & -3 \end{pmatrix}
$$

## Реализация

In [9]:
import cmath
import math


def multiply(A, B):
    return [[sum(A[i][k] * B[k][j] for k in range(len(B))) for j in range(len(B[0]))]
            for i in range(len(A))]


def transpose(A):
    return [list(row) for row in zip(*A)]


def identity(n):
    return [[float(i == j) for j in range(n)] for i in range(n)]


def show(M, name=None, prec=6):
    if name:
        print(f"{name} =")
    for row in M:
        print("  [" + " ".join(f"{(0.0 if abs(v) < 1e-12 else v):{prec + 6}.{prec}f}" for v in row) + " ]")
    print()


def householder(A, j):
    n = len(A)
    norm = math.sqrt(sum(A[i][j] ** 2 for i in range(j, n)))
    v = [0.0] * n
    v[j] = A[j][j] + math.copysign(norm, A[j][j])
    for i in range(j + 1, n):
        v[i] = A[i][j]
    vv = sum(x * x for x in v)
    H = identity(n)
    if vv == 0:
        return H
    for p in range(n):
        for q in range(n):
            H[p][q] -= 2 * v[p] * v[q] / vv
    return H


def qr_decompose(A):
    n = len(A)
    R = [list(map(float, row)) for row in A]
    Q = identity(n)
    for j in range(n - 1):
        H = householder(R, j)
        R = multiply(H, R)
        Q = multiply(Q, H)
    return Q, R


def block_eigenvalues(A, j):
    a, b = A[j][j], A[j][j + 1]
    c, d = A[j + 1][j], A[j + 1][j + 1]
    tr, det = a + d, a * d - b * c
    disc = cmath.sqrt(tr * tr - 4 * det)
    return (tr + disc) / 2, (tr - disc) / 2


def qr(A, eps, max_iter=10_000):
    n = len(A)
    Ak = [list(map(float, row)) for row in A]
    prev_blocks = {}
    history = []

    for k in range(1, max_iter + 1):
        Q, R = qr_decompose(Ak)
        Ak = multiply(R, Q)

        sub = [math.sqrt(sum(Ak[i][j] ** 2 for i in range(j + 1, n))) for j in range(n)]
        history.append(sub)

        eig, j, done = [], 0, True
        blocks = {}
        while j < n:
            if sub[j] <= eps:
                eig.append(Ak[j][j])
                j += 1
            elif j + 1 < n and math.sqrt(sum(Ak[i][j] ** 2 for i in range(j + 2, n))) <= eps:
                pair = block_eigenvalues(Ak, j)
                blocks[j] = pair
                old = prev_blocks.get(j)
                if pair[0].imag == 0:
                    done = False
                elif old is None or max(abs(pair[0] - old[0]), abs(pair[1] - old[1])) > eps:
                    done = False
                eig.extend(pair)
                j += 2
            else:
                done = False
                break
        prev_blocks = blocks
        if done:
            return eig, k, Ak, history
    raise RuntimeError("QR-алгоритм не сошёлся за max_iter итераций")

## Исходные данные

In [10]:
A = [
    [-2, 7, -6],
    [-1, 9, -4],
    [-1, 8, -3],
]
eps = 1e-6

show(A, "A", prec=0)
print(f"eps = {eps}")

A =
  [    -2      7     -6 ]
  [    -1      9     -4 ]
  [    -1      8     -3 ]

eps = 1e-06


## QR-разложение матрицы A

In [11]:
Q, R = qr_decompose(A)
show(Q, "Q")
show(R, "R")
show(multiply(Q, R), "Проверка Q·R")
show(multiply(transpose(Q), Q), "Проверка Q^T·Q")

Q =
  [   -0.816497     0.573068    -0.070186 ]
  [   -0.408248    -0.659028    -0.631676 ]
  [   -0.408248    -0.487108     0.772049 ]

R =
  [    2.449490   -12.655697     7.756718 ]
  [    0.000000    -5.816643     0.659028 ]
  [    0.000000     0.000000     0.631676 ]

Проверка Q·R =
  [   -2.000000     7.000000    -6.000000 ]
  [   -1.000000     9.000000    -4.000000 ]
  [   -1.000000     8.000000    -3.000000 ]

Проверка Q^T·Q =
  [    1.000000     0.000000     0.000000 ]
  [    0.000000     1.000000     0.000000 ]
  [    0.000000     0.000000     1.000000 ]



## Собственные значения QR-алгоритмом

In [12]:
lams, k, Ak, hist = qr(A, eps)

print(f"Итераций: {k}\n")
show(Ak, "A^(k)")
for i, lam in enumerate(lams, 1):
    print(f"lambda_{i} = {fmt(lam)}")

print(f"\nСумма lambda = {sum(lams).real:.6f},  след A = {sum(A[i][i] for i in range(len(A)))}")

Итераций: 20

A^(k) =
  [    4.854102     2.121316   -12.879978 ]
  [   -0.000000    -1.854100     7.912409 ]
  [    0.000000     0.000001     0.999998 ]

lambda_1 =     4.854102
lambda_2 =    -1.854100
lambda_3 =     0.999998

Сумма lambda = 4.000000,  след A = 4
